In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

df = pd.read_csv(
    "retail.csv",
    dtype={
        "StockCode": "string",
        "CustomerID": "string",
    },
)

print("Shape:", df.shape)
print("Missing values:")
print(df.isna().sum())

df.head()

Shape: (397924, 3)
Missing values:
Quantity      0
StockCode     0
CustomerID    0
dtype: int64


,Quantity,StockCode,CustomerID
0,6,85123A,17850.0
1,6,71053,17850.0
2,8,84406B,17850.0
3,6,84029G,17850.0
4,6,84029E,17850.0


In [3]:
features = ["StockCode", "CustomerID"]

cardinality = df[features].nunique()
print(cardinality)

print("\nQuantity distribution:")
print(df["Quantity"].describe(percentiles=[0.5, 0.9, 0.99]))

StockCode     3665
CustomerID    4339
dtype: int64

Quantity distribution:
count    397924.000000
mean         13.021823
std         180.420210
min           1.000000
50%           6.000000
90%          24.000000
99%         120.000000
max       80995.000000
Name: Quantity, dtype: float64


In [4]:
X = df[features].copy()
y = df["Quantity"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 278546
Validation: 59689
Test: 59689


In [ ]:
from category_encoders import TargetEncoder

te_encoder = TargetEncoder(
    cols=features,
    handle_unknown="value",
    handle_missing="value",
)

X_train_cat = X_train[features].astype(object)
X_val_cat = X_val[features].astype(object)
X_test_cat = X_test[features].astype(object)

X_train_te = te_encoder.fit_transform(X_train_cat, y_train)
X_val_te = te_encoder.transform(X_val_cat)
X_test_te = te_encoder.transform(X_test_cat)

print("Encoded training shape:", X_train_te.shape)


Encoded training shape: (278546, 2)


In [ ]:
from sklearn.linear_model import LinearRegression

t_train = np.log1p(y_train.to_numpy(dtype=float))
te_model = LinearRegression()
te_model.fit(X_train_te, t_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[0. ,0.01]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['StockCode','CustomerID']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.736
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


In [11]:
from sklearn.metrics import mean_squared_error

pred_test_log = te_model.predict(X_test_te)
te_test_rmse = float(np.sqrt(mean_squared_error(
    np.log1p(y_test.to_numpy(dtype=float)),
    pred_test_log,
)))

print("Test log RMSE:", te_test_rmse)


Test log RMSE: 0.8981929732879933


In [12]:
import copy
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab = {
    col: {
        value: i + 1
        for i, value in enumerate(X_train[col].unique())
    }
    for col in features
}

def category_indices(X):
    return np.column_stack([
        X[col].map(vocab[col]).fillna(0).to_numpy(dtype=np.int64)
        for col in features
    ])

def embedding_loader(X, y, shuffle=False):
    dataset = TensorDataset(
        torch.from_numpy(category_indices(X)),
        torch.tensor(
            np.log1p(y.to_numpy(dtype=float)),
            dtype=torch.float32,
        ),
    )
    return DataLoader(
        dataset, batch_size=1024, shuffle=shuffle
    )

emb_train_loader = embedding_loader(X_train, y_train, shuffle=True)
emb_val_loader = embedding_loader(X_val, y_val)
emb_test_loader = embedding_loader(X_test, y_test)

In [13]:
class EmbeddingRegressor(nn.Module):
    def __init__(self, n_stock, n_customer, embedding_dim=16):
        super().__init__()

        self.stock_embedding = nn.Embedding(
            n_stock, embedding_dim, padding_idx=0
        )
        self.customer_embedding = nn.Embedding(
            n_customer, embedding_dim, padding_idx=0
        )

        self.net = nn.Sequential(
            nn.Linear(2 * embedding_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        stock = self.stock_embedding(x[:, 0])
        customer = self.customer_embedding(x[:, 1])

        combined = torch.cat([stock, customer], dim=1)
        return self.net(combined).squeeze(-1)


emb_model = EmbeddingRegressor(
    n_stock=len(vocab["StockCode"]) + 1,
    n_customer=len(vocab["CustomerID"]) + 1,
).to(device)

optimizer = torch.optim.Adam(emb_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [14]:
@torch.no_grad()
def embedding_mse(model, loader):
    model.eval()
    total_squared_error = 0.0
    n = 0

    for xb, tb in loader:
        xb = xb.to(device)
        tb = tb.to(device)

        pred = model(xb)
        total_squared_error += (pred - tb).square().sum().item()
        n += len(tb)

    return total_squared_error / n


best_val_mse = float("inf")
best_emb_state = None
best_epoch = None
waiting = 0

max_epochs = 100
patience = 5

for epoch in range(1, max_epochs + 1):
    emb_model.train()

    for xb, tb in emb_train_loader:
        xb = xb.to(device)
        tb = tb.to(device)

        optimizer.zero_grad()
        loss = loss_fn(emb_model(xb), tb)
        loss.backward()
        optimizer.step()

    val_mse = embedding_mse(emb_model, emb_val_loader)

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_emb_state = copy.deepcopy(emb_model.state_dict())
        best_epoch = epoch
        waiting = 0
    else:
        waiting += 1

    print(
        f"Epoch {epoch:3d} | "
        f"Validation log RMSE: {np.sqrt(val_mse):.6f}"
    )

    if waiting >= patience:
        break

emb_model.load_state_dict(best_emb_state)
print("Selected epoch:", best_epoch)

Epoch   1 | Validation log RMSE: 0.961122
Epoch   2 | Validation log RMSE: 0.866085
Epoch   3 | Validation log RMSE: 0.785633
Epoch   4 | Validation log RMSE: 0.720726
Epoch   5 | Validation log RMSE: 0.670350
Epoch   6 | Validation log RMSE: 0.635974
Epoch   7 | Validation log RMSE: 0.611915
Epoch   8 | Validation log RMSE: 0.595930
Epoch   9 | Validation log RMSE: 0.585183
Epoch  10 | Validation log RMSE: 0.577721
Epoch  11 | Validation log RMSE: 0.571689
Epoch  12 | Validation log RMSE: 0.567728
Epoch  13 | Validation log RMSE: 0.563739
Epoch  14 | Validation log RMSE: 0.561178
Epoch  15 | Validation log RMSE: 0.559518
Epoch  16 | Validation log RMSE: 0.556205
Epoch  17 | Validation log RMSE: 0.554871
Epoch  18 | Validation log RMSE: 0.553694
Epoch  19 | Validation log RMSE: 0.552479
Epoch  20 | Validation log RMSE: 0.550459
Epoch  21 | Validation log RMSE: 0.549934
Epoch  22 | Validation log RMSE: 0.548500
Epoch  23 | Validation log RMSE: 0.547016
Epoch  24 | Validation log RMSE: 0

In [15]:
emb_test_rmse = np.sqrt(
    embedding_mse(emb_model, emb_test_loader)
)

comparison = pd.DataFrame({
    "Model": [
        "Target encoding + LR",
        "Embedding neural network",
    ],
    "Test log RMSE": [
        te_test_rmse,
        emb_test_rmse,
    ],
})

comparison

,Model,Test log RMSE
0,Target encoding + LR,0.898193
1,Embedding neural network,0.537094


In [16]:
from sklearn.preprocessing import StandardScaler

te_scaler = StandardScaler()

train_te_scaled = te_scaler.fit_transform(X_train_te)
val_te_scaled = te_scaler.transform(X_val_te)
test_te_scaled = te_scaler.transform(X_test_te)


def combined_loader(X, encoded, y, shuffle=False):
    dataset = TensorDataset(
        torch.from_numpy(category_indices(X)),
        torch.tensor(encoded, dtype=torch.float32),
        torch.tensor(
            np.log1p(y.to_numpy(dtype=float)),
            dtype=torch.float32,
        ),
    )
    return DataLoader(
        dataset, batch_size=1024, shuffle=shuffle
    )


combined_train_loader = combined_loader(
    X_train, train_te_scaled, y_train, shuffle=True
)
combined_val_loader = combined_loader(
    X_val, val_te_scaled, y_val
)
combined_test_loader = combined_loader(
    X_test, test_te_scaled, y_test
)

In [17]:
class CombinedRegressor(nn.Module):
    def __init__(self, n_stock, n_customer, embedding_dim=16):
        super().__init__()

        self.stock_embedding = nn.Embedding(
            n_stock, embedding_dim, padding_idx=0
        )
        self.customer_embedding = nn.Embedding(
            n_customer, embedding_dim, padding_idx=0
        )

        self.net = nn.Sequential(
            nn.Linear(2 * embedding_dim + 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, categories, encoded):
        stock = self.stock_embedding(categories[:, 0])
        customer = self.customer_embedding(categories[:, 1])

        combined = torch.cat(
            [stock, customer, encoded], dim=1
        )
        return self.net(combined).squeeze(-1)


torch.manual_seed(SEED)

combined_model = CombinedRegressor(
    n_stock=len(vocab["StockCode"]) + 1,
    n_customer=len(vocab["CustomerID"]) + 1,
).to(device)

combined_optimizer = torch.optim.Adam(
    combined_model.parameters(), lr=1e-3
)

In [18]:
@torch.no_grad()
def combined_mse(model, loader):
    model.eval()
    total_squared_error = 0.0
    n = 0

    for categories, encoded, target in loader:
        categories = categories.to(device)
        encoded = encoded.to(device)
        target = target.to(device)

        pred = model(categories, encoded)
        total_squared_error += (
            (pred - target).square().sum().item()
        )
        n += len(target)

    return total_squared_error / n


best_combined_val = float("inf")
best_combined_state = None
best_combined_epoch = None
waiting = 0

for epoch in range(1, 101):
    combined_model.train()

    for categories, encoded, target in combined_train_loader:
        categories = categories.to(device)
        encoded = encoded.to(device)
        target = target.to(device)

        combined_optimizer.zero_grad()

        pred = combined_model(categories, encoded)
        loss = loss_fn(pred, target)

        loss.backward()
        combined_optimizer.step()

    val_mse = combined_mse(
        combined_model, combined_val_loader
    )

    if val_mse < best_combined_val:
        best_combined_val = val_mse
        best_combined_state = copy.deepcopy(
            combined_model.state_dict()
        )
        best_combined_epoch = epoch
        waiting = 0
    else:
        waiting += 1

    print(
        f"Epoch {epoch:3d} | "
        f"Validation log RMSE: {np.sqrt(val_mse):.6f}"
    )

    if waiting >= 5:
        break

combined_model.load_state_dict(best_combined_state)
print("Selected epoch:", best_combined_epoch)

Epoch   1 | Validation log RMSE: 0.899331
Epoch   2 | Validation log RMSE: 0.812406
Epoch   3 | Validation log RMSE: 0.723830
Epoch   4 | Validation log RMSE: 0.676030
Epoch   5 | Validation log RMSE: 0.648019
Epoch   6 | Validation log RMSE: 0.629658
Epoch   7 | Validation log RMSE: 0.615761
Epoch   8 | Validation log RMSE: 0.605568
Epoch   9 | Validation log RMSE: 0.596825
Epoch  10 | Validation log RMSE: 0.589931
Epoch  11 | Validation log RMSE: 0.584270
Epoch  12 | Validation log RMSE: 0.579946
Epoch  13 | Validation log RMSE: 0.576370
Epoch  14 | Validation log RMSE: 0.573797
Epoch  15 | Validation log RMSE: 0.570435
Epoch  16 | Validation log RMSE: 0.569402
Epoch  17 | Validation log RMSE: 0.566498
Epoch  18 | Validation log RMSE: 0.565335
Epoch  19 | Validation log RMSE: 0.563847
Epoch  20 | Validation log RMSE: 0.562865
Epoch  21 | Validation log RMSE: 0.561783
Epoch  22 | Validation log RMSE: 0.561579
Epoch  23 | Validation log RMSE: 0.559404
Epoch  24 | Validation log RMSE: 0

In [19]:
combined_test_rmse = np.sqrt(
    combined_mse(combined_model, combined_test_loader)
)

comparison = pd.DataFrame({
    "Model": [
        "Target encoding + LR",
        "Embedding neural network",
        "Target encoding + embeddings",
    ],
    "Test log RMSE": [
        te_test_rmse,
        emb_test_rmse,
        combined_test_rmse,
    ],
})

comparison

,Model,Test log RMSE
0,Target encoding + LR,0.898193
1,Embedding neural network,0.537094
2,Target encoding + embeddings,0.541630
